# 5장 모델 컨텍스트 프로토콜(MCP)

에이전트가 도구를 호출해 외부 정보를 가져오고 작업을 수행합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 5장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents[mcp]

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1. 언어 모델 선택

[5장 기본 실습](chapter05.ipynb)에 이어 Gemma 3를 사용합니다. 프롬프트로 도구를 선택하는 방식을 MCP 서버에 연결합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B(내장 추론과 도구 호출 없음)
llm = LLM(model="gemma3:12b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. MCP로 도구 연결

모델 컨텍스트 프로토콜(Model Context Protocol, MCP)은 도구를 조회하고 호출하는 방법을 표준화합니다. 이번 실습에서는 서버와 클라이언트를 구현하고, MCP 서버가 제공하는 도구를 `TinyAgent`에서 호출합니다.

### MCP 서버

MCP 서버는 외부 서비스나 데이터에 접근하는 기능을 MCP 도구로 제공합니다. 예를 들어 arXiv에 연결된 서버는 논문을 검색하고 내용을 읽는 도구를 제공할 수 있습니다.

![MCP 서버가 외부 서비스를 도구로 제공하는 구조](../images/ch5_mcp_server.png)

서버 코드에서는 함수를 정의하고 도구 데코레이터를 붙인 뒤 표준 입출력(`stdio`)으로 통신하는 서버를 실행합니다. 예제 서버의 `read_markdown`은 URL에서 마크다운 본문을 읽습니다. 클라이언트가 서버를 실행하므로 이 코드를 별도로 실행할 필요는 없습니다. 실제 구현은 [ch5_mcp_server.py](../src/illustrated_agents/chapters/ch5_mcp_server.py)에 있습니다.

```python
import urllib.request

from mcp.server.mcpserver import MCPServer

server = MCPServer("file_reader")

@server.tool()
def read_markdown(path: str) -> str:
    """지정한 URL에서 마크다운 문서를 읽습니다."""
    with urllib.request.urlopen(path) as response:
        return response.read().decode()

def main():
    server.run(transport="stdio")

if __name__ == "__main__":
    main()
```

### MCP 클라이언트

`TinyAgent`는 이 예제의 호스트입니다. 호스트에 있는 클라이언트가 서버에 연결해 도구 목록을 요청하고 도구를 호출합니다.

![호스트의 MCP 클라이언트와 서버의 통신 관계](../images/ch5_mcp_client.png)

먼저 `illustrated-agents` 패키지에 들어 있는 서버 파일의 경로를 구합니다.

In [ ]:
from pathlib import Path
import illustrated_agents

# 노트북에서 MCP를 사용하도록 nest_asyncio 적용
import nest_asyncio
nest_asyncio.apply()

# 도구 모듈에서 MCP 서버를 실행할 때 사용하는 경로
SERVER_PATH = str(Path(illustrated_agents.__file__).parent / "chapters" / "ch5_mcp_server.py")

`list_tools`로 서버에 연결해 사용 가능한 도구를 조회합니다.

In [ ]:
import sys
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession


async def list_tools():
    # MCP 서버 실행 매개변수
    server_params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
    
    # 표준 입출력으로 MCP 서버에 연결
    async with stdio_client(server_params, errlog=None) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            tools = await session.list_tools()
            for tool in tools.tools:
                print(f"Name: {tool.name}")
                print(f"Description: {tool.description}")
                print(f"Parameters: {tool.input_schema}")
                print()

await list_tools()

목록에서 앞서 정의한 함수 이름, 문서 문자열, 인자 정보를 확인할 수 있습니다. 모델은 이 설명을 읽고 도구를 선택하므로 함수의 용도와 인자를 정확하게 적어야 합니다. 토픽 모델링 도구인 BERTopic의 마크다운 문서를 읽어 봅니다.

In [ ]:
async def call_tool():
    # MCP 서버 실행 매개변수
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[SERVER_PATH],
    )
    
    # 표준 입출력으로 MCP 서버에 연결
    async with stdio_client(server_params, errlog=None) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            result = await session.call_tool("read_markdown", {"path": "https://raw.githubusercontent.com/MaartenGr/BERTopic/refs/heads/master/README.md"})
            print(result.content[0].text)

await call_tool()

이제 클라이언트는 서버에서 도구를 조회하고 실행할 수 있습니다. 두 기능을 에이전트에 연결하기 위해 `Tools`를 상속한 `MCPTools` 클래스를 만듭니다.

In [ ]:
import asyncio

from illustrated_agents.tools import Tools

class MCPTools(Tools):
    """Tools를 상속해 MCP 도구를 관리한다."""

    def __init__(self):
        super().__init__()
        self._load_mcp_tools()

    def _get_server_params(self):
        return StdioServerParameters(command=sys.executable, args=[SERVER_PATH])

    def _load_mcp_tools(self):
        """MCP 서버에서 도구 목록을 가져와 등록한다."""

        # MCP 서버에서 도구 목록 조회
        async def fetch():
            async with stdio_client(self._get_server_params(), errlog=None) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    tools = await session.list_tools()
                    return tools.tools

        mcp_tools = asyncio.run(fetch())

        # 부모 클래스의 add_tool로 각 MCP 도구 등록
        for tool in mcp_tools:
            self.add_tool(
                name=tool.name,
                func=self._make_tool_caller(tool.name),
                description=tool.description,
            )

    def _make_tool_caller(self, name: str):
        """MCP 도구를 실행할 함수를 만든다."""

        def caller(**kwargs):
            return asyncio.run(self._call_tool(name, kwargs))

        return caller

    async def _call_tool(self, name: str, args: dict) -> str:
        """MCP 서버의 도구를 호출한다."""
        async with stdio_client(self._get_server_params(), errlog=None) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(name, args)
                return result.content[0].text

다음 코드 설명에서 메서드가 서버를 호출하는 과정을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch5_mcp import run_mcp_tool_annotated; run_mcp_tool_annotated

이번 장의 `TinyAgent`에 `MCPTools`를 전달합니다.

In [ ]:
from illustrated_agents.memory import Memory
from illustrated_agents.chapters.ch5 import TinyAgent

# 서버의 도구 목록을 불러와 MCP 도구 생성
tools = MCPTools()

# 에이전트 생성
memory = Memory()
agent = TinyAgent(llm=llm, tools=tools, memory=memory)

에이전트에게 저자들의 이전 책 저장소에 있는 마크다운 문서를 읽어 달라고 요청합니다. [Hands-On Large Language Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models)

In [ ]:
agent.memory.get_messages()

In [ ]:
query = "Read the file at 'https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/README.md'"
print(agent.run(query))

도구가 적절히 선택되었는지 대화 기록과 결과를 확인합니다.

In [ ]:
agent.memory.get_messages()

---

# 구현한 내용

`Tools`를 상속한 `MCPTools`를 구현해 에이전트에 MCP 도구를 연결했습니다. 기존 도구 클래스와 같은 인터페이스로 사용할 수 있습니다.

In [ ]:
from illustrated_agents.chapters.ch5_mcp import what_we_built; what_we_built

# 다음 실습

[내장 도구 호출](chapter05_native_tool_calling.ipynb)에서 Gemma 4 E4B의 도구 호출을 실습하고, [스킬](chapter05_skills.ipynb)에서 작업 지침을 불러옵니다. [6장](../chapter06/chapter06.ipynb)에서는 추론과 행동을 반복하는 ReAct를 구현합니다.